After extracting all the required information of BACI (in notebook 0.), we format it a little bit in this notebook

In [1]:
import pandas as pd
import sys
sys.path.append('C://Users/11max/PycharmProjects/Regioinvent/src/')
import regioinvent
import sqlite3

Just initialize the Regioinvent class to get all the matching dicts and stuff

In [2]:
self = regioinvent.Regioinvent(bw_project_name='ecoinvent3.12', ecoinvent_database_name='ecoinvent-3.12-cutoff', ecoinvent_version='3.12')

Connecting to the SQL database with trade information

In [3]:
trade_conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/trade_data.db')

Select the international trade data

In [4]:
trade_data = pd.read_sql('SELECT * FROM [Original trade data]', trade_conn)

Select the import data and format it

In [5]:
import_data = trade_data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer', 'exporter']).sum().reset_index()

# go from ISO3 codes to ISO2 codes
import_data.importer = [self.convert_ecoinvent_geos[i] for i in import_data.importer]
import_data.exporter = [self.convert_ecoinvent_geos[i] for i in import_data.exporter]

For documentation, specify on which HS classification the import data is relying on

In [6]:
with_baci_22 = ['290341', '290343', '290349', '854142', '854143', '854159', '441881', '441882', '441883']

import_data.loc[import_data.cmdCode.isin(with_baci_22), 'source'] = 'BACI HS22 202601 (https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37)'
import_data.loc[~import_data.cmdCode.isin(with_baci_22), 'source'] = 'BACI HS17 202601 (https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37)'

Export to the SQL database in a new table called "Import data"

In [10]:
import_data.set_index('cmdCode').to_sql('Import data', trade_conn)

11969181

Select the export data

In [ ]:
export_data = trade_data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()

We then want to calculate the net export balance, and not the pure export data, because we ultimately want to filter out re-exports

In [11]:
# we get the total import data, not including the detail on the origin of the product
total_import_data = trade_data.set_index(['cmdCode', 'refYear', 'importer']).drop(
    ['value (1,000 USD)', 'exporter'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()
export_data = export_data.set_index(['cmdCode', 'refYear', 'exporter'])
total_import_data = total_import_data.set_index(['cmdCode', 'refYear', 'importer'])
total_import_data.index.names = ['cmdCode', 'refYear', 'exporter']
# instead of the exports, we rely on net exports to estimate some of the production volumes
net_exports_data = export_data - total_import_data
net_exports_data = net_exports_data[net_exports_data > 0].fillna(0)
net_exports_data = net_exports_data.reset_index()

For documentation, specify on which HS classification the export data is relying on

In [12]:
with_baci_22 = ['290341', '290343', '290349', '854142', '854143', '854159', '441881', '441882', '441883']

net_exports_data.loc[net_exports_data.cmdCode.isin(with_baci_22), 'source'] = 'BACI HS22 202601 (https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37)'
net_exports_data.loc[~net_exports_data.cmdCode.isin(with_baci_22), 'source'] = 'BACI HS17 202601 (https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37)'

Also, only keep positive net export balances. The rest is fixed to zero, as if they are not exporting the commodity at all.

In [13]:
net_exports_data = net_exports_data.loc[net_exports_data.loc[:,'quantity (t)'] != 0]

Export to the SQL database, in a new table called "Export data"

In [15]:
net_exports_data.loc[net_exports_data.loc[:,'quantity (t)'] != 0].set_index('cmdCode').to_sql('Export data', trade_conn)

145208

In the subsequent notebooks, we estimate production data in physical quantities for all commodities. Indeed, so far our trade data only covers imports and net exports. It does not include domestic production.